# King CRAB 08_09 light-leak measurement

This notebook analyzes both 08_09 oscilloscope runs on channel 4:

- `08_09_Backgroun_Auto`: the auto-trigger acquisition;
- `08_09_Background_Normal`: the normal negative-pulse trigger.

The goals are to measure the pulse-height and charge distributions, determine the timestamp-derived trigger and selected-pulse rates, compare the two acquisition modes, and estimate any rate in excess of the previous channel-4 dark measurement. The Hamamatsu R7378A specification is handled separately because it gives **anode dark current**, not a dark-count rate.

Every conclusion is based on all C4 waveform files. The F1 files are oscilloscope histogram exports and are intentionally excluded from waveform counting. RMS, height, timing, and charge are reconstructed directly from the raw digitized samples after baseline subtraction. No FFT or Savitzky--Golay filter is applied in the primary analysis; filtered results belong only in a systematic cross-check.


In [ ]:
# Locate the repository when Jupyter starts in Notebooks/.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks': PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import SRC.KingCRAB.light_leak as light_leak_helpers
from SRC.KingCRAB.context import configure_module
from SRC.KingCRAB.light_leak import header_timestamps, mean_waveform_from_folder, process_run

from SRC.KingCRAB.digitized import load_digitized_datasets, log_interpolate, log_linear_extrapolation, as_plot_datasets as load_webplotdigitizer_json

from pathlib import Path
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from scipy.stats import ks_2samp
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
ROOT=PROJECT_ROOT
if ROOT.name=='CODE': ROOT=ROOT.parent
DATA=ROOT/'DATA'

RUNS={
 'auto':Path('/Volumes/Untitled/08_09_Backgroun_Auto'),
 'normal':Path('/Volumes/Untitled/08_09_Background_Normal'),
}
for label,path in RUNS.items():
    if not path.exists(): raise FileNotFoundError(f'{label}: mount {path}')

R_OHM=50.0
PRE_END_NS=-20.0
SEARCH_RANGE_NS=(-20.0,80.0)
FIXED_CHARGE_RANGE_NS=(-15.0,35.0)
THRESHOLDS_SIGMA=(3.0,5.0)
HEADER_RE=re.compile(r'^#\d+,[^,]+,(.*)$')

## 1. Reading the segmented LeCroy structure

Each C4 file contains 500 waveform segments following a 504-line header. The third header field is elapsed time from the first trigger in that file. Summing the first-to-last span of each file excludes file-save gaps and provides the same live-time convention used in `PMT_Terminal_Noise_Comparison.ipynb`.

The baseline is the mean before $-20$ ns. Pulse height is the most negative baseline-subtracted sample from $-20$ to $80$ ns. Pulse-region charge uses the zero crossings around that minimum, exactly as in the PMT processing. A fixed $-15$ to $35$ ns signed charge is also retained because it compares the synchronized feature without allowing the integration window to follow a noise minimum.


In [ ]:
configure_module(light_leak_helpers, globals())


auto,auto_timing,t_ns,auto_bank=process_run('auto',RUNS['auto'],save_bank=True)
normal,normal_timing,normal_t_ns,_=process_run('normal',RUNS['normal'])
assert len(t_ns)==len(normal_t_ns)
assert np.isclose(np.median(np.diff(t_ns)),np.median(np.diff(normal_t_ns)),rtol=1e-5)
timing=pd.DataFrame([auto_timing,normal_timing]).set_index('run')
display(timing.round(5))
print(f'Analyzed {len(auto):,} auto and {len(normal):,} normal-trigger waveforms.')

## 2. Charge, height, timing, and thresholded rates

The software thresholds use the mean pre-trigger RMS of each run, matching the global-threshold convention of `PMT_PROCESSING.ipynb`. The selected rate is the selected fraction multiplied by the timestamp-derived acquisition rate. For the auto acquisition this is an occupancy rate at its approximately 60 Hz sampling cadence. For the normal acquisition it is conditional on the oscilloscope hardware trigger.


In [ ]:
frames={'auto':auto,'normal':normal}; rows=[]
for label,frame in frames.items():
    sigma_mean=frame.rms_V.mean(); trig=timing.loc[label,'trigger_rate_Hz']
    for nsigma in THRESHOLDS_SIGMA:
        selected=frame.height_V>nsigma*sigma_mean
        rows.append({'run':label,'threshold_sigma':nsigma,'threshold_mV':nsigma*sigma_mean*1e3,
                     'selected_waveforms':int(selected.sum()),'selected_fraction':selected.mean(),
                     'selected_rate_Hz':selected.mean()*trig,
                     'median_selected_height_mV':frame.loc[selected,'height_V'].median()*1e3,
                     'median_selected_charge_pC':frame.loc[selected,'charge_region_C'].median()*1e12})
selections=pd.DataFrame(rows)
display(selections.round(6))

summary=[]
for label,frame in frames.items():
    summary.append({'run':label,'waveforms':len(frame),'trigger_rate_Hz':timing.loc[label,'trigger_rate_Hz'],
                    'median_rms_mV':frame.rms_V.median()*1e3,'median_height_mV':frame.height_V.median()*1e3,
                    'p99_height_mV':frame.height_V.quantile(.99)*1e3,
                    'median_region_charge_pC':frame.charge_region_C.median()*1e12,
                    'p99_region_charge_pC':frame.charge_region_C.quantile(.99)*1e12,
                    'mean_fixed_charge_pC':frame.charge_fixed_C.mean()*1e12,
                    'mean_total_charge_pC':frame.charge_total_C.mean()*1e12})
run_summary=pd.DataFrame(summary).set_index('run'); display(run_summary.round(5))

fig,ax=plt.subplots(2,2,figsize=(14,9))
colors={'auto':'tab:orange','normal':'tab:blue'}
hr=np.quantile(np.r_[auto.height_V,normal.height_V],[.001,.999])*1e3
qr=np.quantile(np.r_[auto.charge_region_C,normal.charge_region_C],[.001,.999])*1e12
for label,frame in frames.items():
    ax[0,0].hist(frame.height_V*1e3,bins=150,range=hr,histtype='step',density=True,lw=1.6,color=colors[label],label=label)
    ax[0,1].hist(frame.charge_region_C*1e12,bins=150,range=qr,histtype='step',density=True,lw=1.6,color=colors[label],label=label)
    ax[1,0].hist(frame.pulse_time_ns,bins=150,range=SEARCH_RANGE_NS,histtype='step',density=True,lw=1.6,color=colors[label],label=label)
    ax[1,1].hexbin(frame.height_V*1e3,frame.charge_region_C*1e12,gridsize=80,mincnt=1,bins='log',alpha=.55,cmap='Oranges' if label=='auto' else 'Blues')
ax[0,0].set(xlabel='negative pulse height [mV]',ylabel='probability density',title='Pulse-height distribution')
ax[0,1].set(xlabel='zero-crossing charge [pC]',ylabel='probability density',title='Charge distribution')
ax[1,0].set(xlabel='time of negative minimum [ns]',ylabel='probability density',title='Pulse-time distribution')
ax[1,1].set(xlabel='negative pulse height [mV]',ylabel='zero-crossing charge [pC]',title='Charge–height populations')
for a in ax.flat:a.legend() if a is not ax[1,1] else None
fig.tight_layout(); plt.show()

for variable in ['height_V','charge_region_C','pulse_time_ns']:
    result=ks_2samp(auto[variable],normal[variable])
    print(f'{variable}: KS D={result.statistic:.5f}, p={result.pvalue:.3g}')

## 3. Average waveform and the 60 Hz structure

An auto-trigger run is only an unbiased background sample if its trigger times are independent of the waveform. Here both modes have a median interval near 16.67 ms and both averages retain a negative feature at the trigger. This is evidence that the auto sample is synchronized with a roughly 60 Hz structure. It remains valuable as the measured background waveform bank, but it must not be described as uniformly random time sampling.


In [ ]:
configure_module(light_leak_helpers, globals())

# auto_bank is already baseline corrected; normal is streamed once for its mean.
auto_mean=auto_bank.mean(axis=0); normal_mean=mean_waveform_from_folder(RUNS['normal'])
fig,ax=plt.subplots(1,2,figsize=(13,4.3))
ax[0].plot(t_ns,auto_mean*1e3,label='auto',color='tab:orange',lw=1.8)
ax[0].plot(t_ns,normal_mean*1e3,label='normal',color='tab:blue',lw=1.8)
ax[0].axvline(0,color='k',ls=':');ax[0].set(xlabel='time relative to trigger [ns]',ylabel='average voltage [mV]',title='All-waveform averages');ax[0].legend()
for label,folder in RUNS.items():
    all_dt=[]
    for path in sorted(folder.glob('C4C*.txt')):
        s=header_timestamps(path)
        if len(s)>1:all_dt.extend(np.diff(s)*1e3)
    ax[1].hist(all_dt,bins=150,range=(0,25),histtype='step',density=True,lw=1.7,label=label,color=colors[label])
ax[1].axvline(1000/60,color='k',ls=':',label='16.67 ms');ax[1].set(xlabel='inter-trigger interval [ms]',ylabel='probability density',title='Both acquisitions contain a 60 Hz cadence');ax[1].legend()
fig.tight_layout();plt.show()

## 4. Excess above dark response

Two references must not be conflated:

1. The earlier channel-4 powered dark run supplies a directly measured hardware-trigger and software-selected count rate under its own threshold conditions.
2. The digitized R7378A DATA provide voltage-dependent **anode dark current** and gain curves. Current is not a dark-count rate. Dividing the DATA-interpolated current by the DATA-interpolated SPE charge gives only an SPE-equivalent current rate, not the number of discrete dark pulses that the oscilloscope should trigger on.

The measured light-leak excess is conservatively defined as the positive part of the 08_09 rate minus the matching earlier channel-4 dark rate. A negative difference is reported as no resolved positive excess rather than as negative light.


## 5. Inputs retained in the notebook

The complete feature tables and the baseline-subtracted auto-trigger waveform bank remain available as the in-memory objects `auto`, `normal`, and `auto_bank`. The notebook deliberately does not create CSV, PNG, or NPZ side products. The cosmogenic notebook reads the original 08_09 waveforms directly, so its background model does not depend on exported intermediate files.

The bank is still conditional on the observed 60 Hz synchronization. It is therefore a measured 08_09 background model, not proof of stationary random background.


In [ ]:
print('No CSV, PNG, or NPZ files were written; plots and tables remain embedded here.')
print(f'In-memory waveform bank shape: {auto_bank.shape}; array size {auto_bank.nbytes/1e6:.1f} MB')
print('\nConclusion:')
if comparison['resolved positive excess [Hz]'].max()>0:
    print('A positive excess is resolved by at least one stated rate definition.')
else:
    print('No positive 08_09 excess is resolved above the earlier channel-4 dark reference.')
print('Both 08_09 modes show a synchronized ~60 Hz waveform feature; treat it as measured background in injection studies.')